# ExoHunter-ML
## Detecção de sinais de exoplanetas em curvas de luz com Machine Learning

**Disciplina:** Engenharia de Aprendizado de Máquina  
**Trilha:** A — Aprendizado Supervisionado (Classificação)

### Objetivo

Desenvolver uma solução completa e reproduzível de Machine Learning para classificar curvas de luz e identificar observações com sinais associados à presença de exoplanetas.

O pipeline contempla:

**ingestão → EDA → preparação → engenharia de atributos → divisão dos dados → ajuste de hiperparâmetros → treinamento → avaliação → explicabilidade → demonstração → operacionalização.**

> O projeto não é apenas um classificador. Ele demonstra o ciclo completo de Engenharia de Machine Learning, desde os dados até uma proposta de monitoramento em produção.

## 1. Ambiente e reprodutibilidade

As dependências são instaladas no próprio notebook e uma semente aleatória é definida para tornar os experimentos reproduzíveis.

Também evitei caminhos locais: o dataset será carregado diretamente de uma fonte pública.

In [ ]:
!pip -q install numpy pandas scipy scikit-learn matplotlib seaborn shap

: 

In [ ]:
import os
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import skew, kurtosis

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, classification_report,
    ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print(f"Random seed: {SEED}")

## 2. Ingestão dos dados

Utilizei o **Kepler Labelled Time Series Data**, um conjunto público de séries temporais do telescópio Kepler.

Cada observação contém uma variável `LABEL` e vários pontos `FLUX.*`, que representam a curva de luz.

A variável original é convertida para classificação binária:

- **0 — sem exoplaneta**
- **1 — com exoplaneta**

In [ ]:
# Download e carregamento do dataset público do Kaggle

!pip -q install kagglehub

import kagglehub
import os

DATASET_ID = "keplersmachines/kepler-labelled-time-series-data"

dataset_path = kagglehub.dataset_download(DATASET_ID)

print("Dataset baixado em:")
print(dataset_path)

print("\nArquivos disponíveis:")
print(os.listdir(dataset_path))

In [ ]:
# Carregamento dos conjuntos de treinamento e teste

TRAIN_PATH = os.path.join(dataset_path, "exoTrain.csv")
TEST_PATH = os.path.join(dataset_path, "exoTest.csv")

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

flux_columns = [c for c in train.columns if c.startswith("FLUX.")]

print("Treinamento:", train.shape)
print("Teste:", test.shape)
print("Pontos por curva:", len(flux_columns))

In [ ]:
display(train.head())

## 3. Validação e análise exploratória

Antes da modelagem verifiquei a qualidade básica dos dados e observei a distribuição das classes.

A EDA completa foi realizada no notebook de laboratório. Nesta versão final mantive apenas as análises necessárias para documentar e justificar o pipeline.

In [ ]:
quality = pd.DataFrame({
    "Indicador": [
        "Amostras de treino", "Amostras de teste",
        "Pontos por curva",
        "Valores ausentes - treino", "Valores ausentes - teste",
        "Valores infinitos - treino", "Valores infinitos - teste",
        "Duplicatas - treino", "Duplicatas - teste"
    ],
    "Valor": [
        len(train), len(test), len(flux_columns),
        int(train.isna().sum().sum()),
        int(test.isna().sum().sum()),
        int(np.isinf(train[flux_columns].to_numpy()).sum()),
        int(np.isinf(test[flux_columns].to_numpy()).sum()),
        int(train.duplicated().sum()),
        int(test.duplicated().sum())
    ]
})
display(quality)

A base não apresenta valores ausentes, valores infinitos ou linhas duplicadas, ou seja, não é necessário nenhum tratamento de limpeza antes da engenharia de atributos. Essa verificação é importante porque a robustez do pipeline depende de dados consistentes desde a entrada.

In [ ]:
counts = train["LABEL"].value_counts().sort_index()

plt.figure(figsize=(7,4))
plt.bar(["Classe 1", "Classe 2"], counts.values)
plt.ylabel("Quantidade")
plt.title("Distribuição das classes")
plt.show()

display(pd.DataFrame({
    "Classe": counts.index,
    "Quantidade": counts.values,
    "Percentual": (counts / counts.sum() * 100).round(2).values
}))

### Desbalanceamento de classes

A classe positiva (com exoplaneta) representa uma fração muito pequena do total de observações. Esse desbalanceamento é esperado no domínio: trânsitos planetários são eventos raros dentro do conjunto de curvas observadas.

Essa característica tem duas consequências diretas mais adiante no notebook:

1. **Accuracy sozinha não é uma métrica confiável** — um modelo que sempre prevê "sem exoplaneta" teria accuracy alta e seria inútil. Por isso o F1-score e o Recall recebem mais atenção na avaliação.
2. **O Random Forest é treinado com `class_weight="balanced"`**, para que a classe minoritária não seja ignorada durante o ajuste das árvores.

In [ ]:
negative_curve = train.loc[train["LABEL"] == 1, flux_columns].iloc[0].to_numpy(float)
positive_curve = train.loc[train["LABEL"] == 2, flux_columns].iloc[0].to_numpy(float)

fig, axes = plt.subplots(2, 1, figsize=(14,7), sharex=True)

axes[0].plot(negative_curve, linewidth=0.8)
axes[0].set_title("Exemplo — sem exoplaneta")
axes[0].set_ylabel("Fluxo")

axes[1].plot(positive_curve, linewidth=0.8)
axes[1].set_title("Exemplo — com exoplaneta")
axes[1].set_xlabel("Índice temporal")
axes[1].set_ylabel("Fluxo")

plt.tight_layout()
plt.show()

### Interpretação da EDA

As curvas possuem muitos pontos temporais e podem apresentar escalas e valores extremos diferentes.

Por isso, em vez de utilizar diretamente todos os pontos da curva como entrada do Random Forest, realizei **engenharia de atributos estatísticos**.

> A decisão é transformar cada curva em um vetor compacto de características. Isso reduz a dimensionalidade e fornece ao modelo medidas capazes de representar tendência central, dispersão, extremos e formato da distribuição.

## 4. Preparação da variável-alvo

A variável original possui:

- `LABEL = 1`: sem exoplaneta;
- `LABEL = 2`: com exoplaneta.

A codificação utilizada é:

**LABEL 2 → 1** e **LABEL 1 → 0**.

A classe positiva, portanto, representa a presença de um exoplaneta.

In [ ]:
X_full = train[flux_columns].copy()
X_test = test[flux_columns].copy()

y_full = (train["LABEL"] == 2).astype(int)
y_test = (test["LABEL"] == 2).astype(int)

print("Distribuição do treino:")
display(y_full.value_counts().sort_index())

print("Distribuição do teste:")
display(y_test.value_counts().sort_index())

## 5. Engenharia de atributos

Cada curva é convertida em **15 atributos estatísticos**:

- tendência central: `mean`, `median`;
- dispersão: `std`, `mad`, `range`;
- extremos: `min`, `max`;
- quantis: `q01`, `q05`, `q25`, `q75`, `q95`, `q99`;
- formato da distribuição: `skewness`, `kurtosis`.

O MAD (Median Absolute Deviation) é uma medida robusta de dispersão e complementa o desvio-padrão.

Assim, uma curva com milhares de pontos passa a ser representada por apenas 15 variáveis.

In [ ]:
FEATURE_NAMES = [
    "median", "mad", "mean", "std", "min", "max", "range",
    "q01", "q05", "q25", "q75", "q95", "q99",
    "skewness", "kurtosis"
]

def extract_features(data):
    """Extrai os mesmos 15 atributos para qualquer curva de luz."""
    if isinstance(data, np.ndarray):
        arr = np.asarray(data, dtype=float)
        if arr.ndim == 1:
            arr = arr.reshape(1, -1)
        flux = arr
    else:
        flux = data[flux_columns].to_numpy(dtype=float)

    rows = []
    for x in flux:
        median = np.median(x)
        rows.append({
            "median": median,
            "mad": np.median(np.abs(x - median)),
            "mean": np.mean(x),
            "std": np.std(x),
            "min": np.min(x),
            "max": np.max(x),
            "range": np.ptp(x),
            "q01": np.quantile(x, .01),
            "q05": np.quantile(x, .05),
            "q25": np.quantile(x, .25),
            "q75": np.quantile(x, .75),
            "q95": np.quantile(x, .95),
            "q99": np.quantile(x, .99),
            "skewness": skew(x, bias=False),
            "kurtosis": kurtosis(x, bias=False)
        })

    return pd.DataFrame(rows, columns=FEATURE_NAMES)

In [ ]:
X_features = extract_features(X_full)
X_test_features = extract_features(X_test)

print("Features de treino:", X_features.shape)
print("Features de teste:", X_test_features.shape)
display(X_features.head())

### Verificação da transformação

A mesma função `extract_features()` é utilizada no treinamento, validação, teste e demonstração.

Isso é fundamental para evitar inconsistências entre a entrada usada para treinar o modelo e a entrada usada posteriormente para fazer previsões.

In [ ]:
feature_check = X_features.describe().T
feature_check["missing"] = X_features.isna().sum().values
feature_check["infinite"] = np.isinf(X_features.to_numpy()).sum(axis=0)
display(feature_check)

## 6. Separação treino / validação / teste

O dataset de treinamento é dividido em:

- **80% treino:** ajuste do modelo;
- **20% validação:** decisões durante o desenvolvimento.

O `exoTest.csv` permanece isolado para a avaliação final.

A estratificação preserva aproximadamente a proporção das classes nos dois subconjuntos.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_features,
    y_full,
    test_size=0.20,
    stratify=y_full,
    random_state=SEED
)

print("Treino:", X_train.shape)
print("Validação:", X_val.shape)
print("Teste final:", X_test_features.shape)

## 7. Modelagem e ajuste de hiperparâmetros

O modelo escolhido é o **Random Forest**.

Ele é adequado ao problema porque trabalha bem com atributos numéricos, captura relações não lineares, não exige padronização para seu funcionamento e fornece medidas de importância das características.

Para atender à etapa de engenharia de ML, os hiperparâmetros não são escolhidos arbitrariamente: usamos `RandomizedSearchCV` com validação cruzada de 3 folds.

O objetivo de otimização é o **F1-score**, que equilibra Precision e Recall.

In [ ]:
rf = RandomForestClassifier(
    random_state=SEED,
    class_weight="balanced",
    n_jobs=-1
)

param_distributions = {
    "n_estimators": [200, 300, 400],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None]
}

search = RandomizedSearchCV(
    rf,
    param_distributions=param_distributions,
    n_iter=12,
    scoring="f1",
    cv=3,
    random_state=SEED,
    n_jobs=-1,
    verbose=1
)

search.fit(X_train, y_train)
best_rf = search.best_estimator_

print("Melhores hiperparâmetros:")
print(search.best_params_)
print(f"F1 médio na validação cruzada: {search.best_score_:.4f}")

## 8. Avaliação na validação

Foi avaliado o melhor modelo utilizando:

- **Accuracy:** proporção total de acertos;
- **Precision:** confiabilidade das previsões positivas;
- **Recall:** capacidade de encontrar os casos positivos;
- **F1-score:** equilíbrio entre Precision e Recall;
- **ROC-AUC:** capacidade de separar as classes considerando diferentes limiares;
- **Average Precision:** resumo do desempenho Precision-Recall.

Para detecção de exoplanetas, Recall é especialmente relevante porque um falso negativo representa uma possível candidata que deixou de ser detectada.

In [ ]:
y_val_proba = best_rf.predict_proba(X_val)[:, 1]
y_val_pred = best_rf.predict(X_val)

validation_metrics = {
    "Accuracy": accuracy_score(y_val, y_val_pred),
    "Precision": precision_score(y_val, y_val_pred, zero_division=0),
    "Recall": recall_score(y_val, y_val_pred, zero_division=0),
    "F1-score": f1_score(y_val, y_val_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_val, y_val_proba),
    "Average Precision": average_precision_score(y_val, y_val_proba)
}

display(pd.DataFrame({
    "Métrica": validation_metrics.keys(),
    "Valor": validation_metrics.values()
}).round(4))

A tabela acima resume o desempenho em números — a matriz de confusão a seguir mostra *onde* o modelo acerta e erra: quantos falsos negativos (candidatas perdidas) e falsos positivos (alarmes indevidos) ocorreram na validação.

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_val,
    y_val_pred,
    display_labels=["Sem exoplaneta", "Com exoplaneta"],
    values_format="d"
)
plt.title("Matriz de confusão — validação")
plt.show()

> Não repare apenas na acurácia. A Precision responde “quantos dos candidatos detectados realmente são positivos?” e Recall responde “quantos dos positivos existentes conseguimos encontrar?”. O F1 resume esse equilíbrio.

## 9. Avaliação final no conjunto de teste

O conjunto de teste não participou do ajuste dos hiperparâmetros.

Somente agora farei a avaliação final, obtendo uma estimativa mais imparcial do desempenho em observações não utilizadas durante o desenvolvimento.

In [ ]:
y_test_proba = best_rf.predict_proba(X_test_features)[:, 1]
y_test_pred = best_rf.predict(X_test_features)

test_metrics = {
    "Accuracy": accuracy_score(y_test, y_test_pred),
    "Precision": precision_score(y_test, y_test_pred, zero_division=0),
    "Recall": recall_score(y_test, y_test_pred, zero_division=0),
    "F1-score": f1_score(y_test, y_test_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_test_proba),
    "Average Precision": average_precision_score(y_test, y_test_proba)
}

display(pd.DataFrame({
    "Métrica": test_metrics.keys(),
    "Resultado": test_metrics.values()
}).round(4))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_test_pred,
    display_labels=["Sem exoplaneta", "Com exoplaneta"],
    values_format="d"
)
plt.title("Matriz de confusão — teste")
plt.show()

print(classification_report(
    y_test,
    y_test_pred,
    target_names=["Sem exoplaneta", "Com exoplaneta"],
    digits=4
))

## 10. Importância das características

O Random Forest permite observar quais atributos foram mais utilizados nas decisões das árvores.

Essa é uma primeira forma de interpretação global do modelo. Em seguida, o SHAP será utilizado para explicar as contribuições das características de maneira mais detalhada.

In [ ]:
feature_importance = pd.Series(
    best_rf.feature_importances_,
    index=FEATURE_NAMES
).sort_values(ascending=False)

display(feature_importance.rename("Importância").to_frame().round(4))

plt.figure(figsize=(9,6))
feature_importance.sort_values().plot(kind="barh")
plt.xlabel("Importância")
plt.ylabel("Atributo")
plt.title("Importância das características — Random Forest")
plt.tight_layout()
plt.show()

## 11. Explicabilidade com SHAP

**SHAP (SHapley Additive exPlanations)** estima a contribuição de cada característica para uma previsão.

Neste projeto, a análise global responde:

> **Quais propriedades estatísticas mais influenciam o modelo na identificação de uma curva como pertencente à classe “com exoplaneta”?**

Isso atende à etapa de explicabilidade do trabalho e torna o modelo mais interpretável.

> obs.: Dependendo da versão da biblioteca SHAP, o retorno para um classificador binário muda de formato: versões mais antigas devolvem uma lista com um array por classe ([shap_classe0, shap_classe1]); versões mais recentes devolvem um único array 3D (amostras x atributos x classes). O bloco abaixo trata os dois casos e sempre extrai os valores SHAP referentes à classe positiva ("com exoplaneta").

In [ ]:
import shap

explainer = shap.TreeExplainer(best_rf)
shap_values = explainer.shap_values(X_val)

if isinstance(shap_values, list):
    shap_positive = shap_values[1]
elif np.asarray(shap_values).ndim == 3:
    shap_positive = np.asarray(shap_values)[:, :, 1]
else:
    shap_positive = np.asarray(shap_values)

print("SHAP calculado.")
print("Formato:", np.asarray(shap_positive).shape)

In [ ]:
shap.summary_plot(
    shap_positive,
    X_val,
    feature_names=FEATURE_NAMES,
    show=True
)

In [ ]:
shap_importance = pd.Series(
    np.abs(shap_positive).mean(axis=0),
    index=FEATURE_NAMES
).sort_values(ascending=False)

display(shap_importance.rename("Mean |SHAP|").to_frame().round(5))

### Como interpretar o SHAP

- valores positivos empurram a previsão em direção à classe **com exoplaneta**;
- valores negativos empurram a previsão em direção à classe **sem exoplaneta**;
- quanto maior o valor absoluto do SHAP, maior a influência daquela característica naquela previsão.

> A importância tradicional responde “quais variáveis o modelo usa mais?”. O SHAP permite avançar para “como essas variáveis influenciam as previsões?”.

## 12. Demonstração funcional

Agora executamos uma inferência completa sobre uma curva do conjunto de teste que não participou do treinamento.

Fluxo:

**curva de luz → extração de atributos → Random Forest → probabilidade → decisão.**

O limiar de 0,50 é utilizado aqui para manter a demonstração alinhada ao comportamento padrão do classificador.

In [ ]:
def predict_curve(curve, model=best_rf, threshold=0.50):
    """Executa o pipeline completo de inferência para uma nova curva."""
    features = extract_features(np.asarray(curve, dtype=float))
    probability = model.predict_proba(features)[0, 1]
    prediction = int(probability >= threshold)

    return {
        "probabilidade_exoplaneta": probability,
        "predicao": prediction,
        "features": features
    }

demo_index = 0
curve_demo = X_test.iloc[demo_index].to_numpy(dtype=float)
demo_result = predict_curve(curve_demo)

print("=== DEMONSTRAÇÃO DA INFERÊNCIA ===")
print(f"Índice da amostra: {demo_index}")
print(f"Probabilidade: {demo_result['probabilidade_exoplaneta']:.2%}")
print(
    "Decisão:",
    "COM EXOPLANETA" if demo_result["predicao"] == 1
    else "SEM EXOPLANETA"
)

In [ ]:
plt.figure(figsize=(14,4))
plt.plot(curve_demo, linewidth=0.8)
plt.xlabel("Índice temporal")
plt.ylabel("Fluxo")
plt.title("Curva utilizada na demonstração")
plt.grid(alpha=0.25)
plt.show()

A demonstração utiliza exatamente a mesma transformação aplicada no treinamento.

Em produção, `predict_curve()` poderia receber uma nova curva produzida pelo sistema de aquisição, gerar automaticamente as 15 características e retornar a probabilidade e a decisão.

## 13. Operacionalização e monitoramento

O modelo ainda não está implantado em produção neste trabalho. Entretanto, uma solução de ML precisa prever como será acompanhada após o treinamento.

O monitoramento deve observar:

- **Data Drift:** mudanças na distribuição das características;
- **Prediction Drift:** mudanças na distribuição das probabilidades;
- **Performance:** Precision, Recall e F1 quando novos rótulos estiverem disponíveis;
- **Falsos negativos:** especialmente importantes para uma tarefa de detecção;
- **Volume de dados:** alterações no fluxo de observações.

In [ ]:
monitoramento = pd.DataFrame({
    "Indicador": [
        "Distribuição das features",
        "Distribuição das probabilidades",
        "Recall",
        "Precision",
        "F1-score",
        "Taxa de falsos negativos",
        "Volume de novas observações"
    ],
    "Objetivo": [
        "Detectar data drift",
        "Detectar prediction drift",
        "Monitorar detecção de positivos",
        "Monitorar falsos alarmes",
        "Acompanhar desempenho geral",
        "Controlar o principal risco de detecção",
        "Verificar alterações no fluxo de dados"
    ],
    "Ação": [
        "Investigar mudança nos dados",
        "Investigar alteração no comportamento",
        "Avaliar retreinamento",
        "Investigar qualidade e modelo",
        "Comparar com baseline",
        "Investigar prioritariamente",
        "Verificar pipeline de aquisição"
    ]
})

display(monitoramento)

### Estratégia de retreinamento

Uma estratégia de manutenção pode seguir:

**coleta → validação dos dados → monitoramento de drift → avaliação → retreinamento → validação do novo modelo → substituição se aprovado.**

O retreinamento deve ser acionado por evidências de degradação ou pela disponibilidade de uma quantidade relevante de novos dados rotulados.

> Operacionalizar não significa apenas “colocar o modelo em produção”. É necessário acompanhar os dados, as previsões e o desempenho ao longo do tempo.

# 14. Conclusão

O ExoHunter-ML implementa uma solução supervisionada para classificação de curvas de luz do telescópio Kepler.

O pipeline desenvolvido contempla:

1. ingestão de uma base pública;
2. análise exploratória;
3. validação e preparação dos dados;
4. engenharia de 15 atributos estatísticos;
5. separação treino/validação/teste;
6. ajuste de hiperparâmetros;
7. Random Forest;
8. avaliação por múltiplas métricas;
9. explicabilidade por importância de atributos e SHAP;
10. demonstração funcional de inferência;
11. proposta de monitoramento, drift e retreinamento.

O resultado é uma solução de Machine Learning documentada de ponta a ponta, alinhada ao ciclo de vida solicitado na disciplina.

# Resumo

### Problema
Identificar sinais associados a exoplanetas em curvas de luz do Kepler.

### Dados
Dataset público com curvas de fluxo e rótulos de presença/ausência de exoplaneta.

### Engenharia de atributos
15 estatísticas representam tendência central, dispersão, extremos, quantis e formato da curva.

### Modelo
Random Forest com `RandomizedSearchCV` e validação cruzada.

### Avaliação
Accuracy, Precision, Recall, F1, ROC-AUC, Average Precision e matriz de confusão.

### Explicabilidade
Importância das características + SHAP.

### Demonstração
Uma curva não utilizada no treinamento passa pelo mesmo pipeline e recebe probabilidade e decisão.

### Produção
Monitoramento de data drift, prediction drift, desempenho e falsos negativos, com retreinamento quando houver evidência de degradação.

> O projeto demonstra o ciclo completo de Engenharia de Machine Learning, e não somente o treinamento de um modelo.